<a href="https://colab.research.google.com/github/yadhnyarpatil5/OOP-Library-Management-System/blob/main/OOP_Library_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
from datetime import datetime, timedelta

# =========================
# BOOK CLASS
# =========================

class Book:
    def __init__(self, title, author, isbn, year):
        self.title = title
        self.author = author
        self.isbn = isbn
        self.year = year
        self.available = True
        self.borrowed_by = None
        self.due_date = None

    def check_out(self, member_id):
        if not self.available:
            return False

        self.available = False
        self.borrowed_by = member_id
        self.due_date = (datetime.now() + timedelta(days=14)).strftime("%Y-%m-%d")
        return True

    def return_book(self):
        self.available = True
        self.borrowed_by = None
        self.due_date = None

    def to_dict(self):
        return {
            "title": self.title,
            "author": self.author,
            "isbn": self.isbn,
            "year": self.year,
            "available": self.available,
            "borrowed_by": self.borrowed_by,
            "due_date": self.due_date
        }

    @classmethod
    def from_dict(cls, data):
        book = cls(
            data["title"],
            data["author"],
            data["isbn"],
            data["year"]
        )
        book.available = data["available"]
        book.borrowed_by = data["borrowed_by"]
        book.due_date = data["due_date"]
        return book


# =========================
# MEMBER CLASS
# =========================

class Member:
    MAX_BOOKS = 5

    def __init__(self, name, member_id):
        self.name = name
        self.member_id = member_id
        self.borrowed_books = []

    def borrow_book(self, isbn):
        if len(self.borrowed_books) >= self.MAX_BOOKS:
            return False

        self.borrowed_books.append(isbn)
        return True

    def return_book(self, isbn):
        if isbn in self.borrowed_books:
            self.borrowed_books.remove(isbn)

    def to_dict(self):
        return {
            "name": self.name,
            "member_id": self.member_id,
            "borrowed_books": self.borrowed_books
        }

    @classmethod
    def from_dict(cls, data):
        member = cls(
            data["name"],
            data["member_id"]
        )
        member.borrowed_books = data["borrowed_books"]
        return member


# =========================
# LIBRARY CLASS
# =========================

class Library:
    def __init__(self):
        self.books = {}
        self.members = {}

    # BOOK METHODS
    def add_book(self, book):
        self.books[book.isbn] = book

    def remove_book(self, isbn):
        if isbn in self.books:
            del self.books[isbn]

    def find_book(self, keyword):
        results = []

        for book in self.books.values():
            if (keyword.lower() in book.title.lower()
                    or keyword.lower() in book.author.lower()
                    or keyword == book.isbn):
                results.append(book)

        return results

    # MEMBER METHODS
    def register_member(self, member):
        self.members[member.member_id] = member

    # BORROW
    def borrow_book(self, member_id, isbn):

        if member_id not in self.members:
            print("Member not found")
            return

        if isbn not in self.books:
            print("Book not found")
            return

        member = self.members[member_id]
        book = self.books[isbn]

        if not member.borrow_book(isbn):
            print("Borrow limit reached")
            return

        if book.check_out(member_id):
            print("Book borrowed successfully")
        else:
            print("Book unavailable")

    # RETURN
    def return_book(self, member_id, isbn):

        if member_id not in self.members:
            return

        if isbn not in self.books:
            return

        member = self.members[member_id]
        book = self.books[isbn]

        member.return_book(isbn)

        overdue_days = 0

        if book.due_date:
            due = datetime.strptime(book.due_date, "%Y-%m-%d")

            if datetime.now() > due:
                overdue_days = (datetime.now() - due).days

        book.return_book()

        if overdue_days > 0:
            print(f"Returned. Overdue by {overdue_days} days.")
        else:
            print("Book returned successfully.")

    # DISPLAY
    def show_books(self):

        if not self.books:
            print("No books available")
            return

        for book in self.books.values():
            status = "Available" if book.available else "Borrowed"

            print("\n--------------------")
            print("Title :", book.title)
            print("Author:", book.author)
            print("ISBN  :", book.isbn)
            print("Year  :", book.year)
            print("Status:", status)

    def show_members(self):

        if not self.members:
            print("No members registered")
            return

        for member in self.members.values():
            print("\n--------------------")
            print("Name:", member.name)
            print("ID  :", member.member_id)
            print("Books Borrowed:", len(member.borrowed_books))

    # SAVE DATA
    def save_data(self):

        books_data = {
            isbn: book.to_dict()
            for isbn, book in self.books.items()
        }

        members_data = {
            mid: member.to_dict()
            for mid, member in self.members.items()
        }

        with open("books.json", "w") as f:
            json.dump(books_data, f, indent=4)

        with open("members.json", "w") as f:
            json.dump(members_data, f, indent=4)

        print("Data saved successfully")

    # LOAD DATA
    def load_data(self):

        if os.path.exists("books.json"):
            with open("books.json", "r") as f:
                books_data = json.load(f)

            for isbn, data in books_data.items():
                self.books[isbn] = Book.from_dict(data)

        if os.path.exists("members.json"):
            with open("members.json", "r") as f:
                members_data = json.load(f)

            for mid, data in members_data.items():
                self.members[mid] = Member.from_dict(data)


# =========================
# MAIN PROGRAM
# =========================

library = Library()
library.load_data()

while True:

    print("\n")
    print("=" * 40)
    print("LIBRARY MANAGEMENT SYSTEM")
    print("=" * 40)

    print("1. Add Book")
    print("2. Register Member")
    print("3. Borrow Book")
    print("4. Return Book")
    print("5. Search Book")
    print("6. View All Books")
    print("7. View All Members")
    print("8. Save Data")
    print("9. Exit")

    choice = input("Enter choice: ")

    if choice == "1":

        title = input("Title: ")
        author = input("Author: ")
        isbn = input("ISBN: ")
        year = input("Year: ")

        library.add_book(
            Book(title, author, isbn, year)
        )

        print("Book added successfully")

    elif choice == "2":

        name = input("Member Name: ")
        member_id = input("Member ID: ")

        library.register_member(
            Member(name, member_id)
        )

        print("Member registered successfully")

    elif choice == "3":

        member_id = input("Member ID: ")
        isbn = input("ISBN: ")

        library.borrow_book(member_id, isbn)

    elif choice == "4":

        member_id = input("Member ID: ")
        isbn = input("ISBN: ")

        library.return_book(member_id, isbn)

    elif choice == "5":

        keyword = input("Search: ")

        results = library.find_book(keyword)

        if results:
            print("\nSearch Results")
            for book in results:
                print(f"{book.title} - {book.author}")
        else:
            print("No books found")

    elif choice == "6":
        library.show_books()

    elif choice == "7":
        library.show_members()

    elif choice == "8":
        library.save_data()

    elif choice == "9":
        library.save_data()
        print("Thank You!")
        break

    else:
        print("Invalid choice")



LIBRARY MANAGEMENT SYSTEM
1. Add Book
2. Register Member
3. Borrow Book
4. Return Book
5. Search Book
6. View All Books
7. View All Members
8. Save Data
9. Exit
Enter choice: 1
Title: Aarambhika
Author: Nelson Mandela
ISBN: 8757788
Year: 2016
Book added successfully


LIBRARY MANAGEMENT SYSTEM
1. Add Book
2. Register Member
3. Borrow Book
4. Return Book
5. Search Book
6. View All Books
7. View All Members
8. Save Data
9. Exit
Enter choice: 7
No members registered


LIBRARY MANAGEMENT SYSTEM
1. Add Book
2. Register Member
3. Borrow Book
4. Return Book
5. Search Book
6. View All Books
7. View All Members
8. Save Data
9. Exit
